In [1]:
# Core Python utilities
import os
import re
import time
import json
import textwrap
from pathlib import Path
from typing import List, Dict, Tuple, Any, Optional

# Numerical computation
import numpy as np

# PDF reading
import pymupdf # PyMuPDF

# Jupyter Notebook parsing
import nbformat

# Embedding model
from sentence_transformers import SentenceTransformer

# Qdrant vector database
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
)

# Hugging Face generation model
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device available:', device)
if device == 'cuda':
  print('GPU name:', torch.cuda.get_device_name(0))
else:
  print('GPU not available. The note book will run on CPU')

device available: cpu
GPU not available. The note book will run on CPU


In [3]:
def print_wrapped(text: str, width: int = 100) -> None:
    """
    Print long text in a readable wrapped format.

    Parameters
    ----------
    text : str
        Text to print.
    width : int
        Maximum line width.
    """
    print(textwrap.fill(text, width=width))

In [4]:
sample_text = (
    "Retrieval-Augmented Generation first retrieves relevant context from an external source "
    "and then uses a generative model to produce an answer grounded in that retrieved context."
)

print_wrapped(sample_text, width=80)

Retrieval-Augmented Generation first retrieves relevant context from an external
source and then uses a generative model to produce an answer grounded in that
retrieved context.


In [5]:
import os

code_file_names = os.listdir()

print("Files in current directory:")

for file_name in code_file_names:
    print(file_name)

Files in current directory:
RAG_Codefile_Retrival.ipynb
RAG_Codefile_Retrival.ipynb.bak
.DS_Store
Retrival_Ext_QnA_Complete.ipynb
daedalus_qdrant


In [6]:
import os

file_name = "corpus/Retrival_Ext_QnA_Complete.ipynb"

if os.path.exists(file_name):
    print("File found:", file_name)
else:
    print("File not found:", file_name)

File found: Retrival_Ext_QnA_Complete.ipynb


In [7]:
# ------------------------------------------------------------
# Select the code file to ingest.
#
# FIX: this used to be `code_file_path = code_file_names[1]`.
# os.listdir() order is arbitrary and the directory also contains
# .DS_Store, the ./daedalus_qdrant store, and this notebook itself,
# so index 1 silently pointed at the wrong entry. Select by name
# and validate it instead.
# ------------------------------------------------------------

CODE_FILE_NAME = "corpus/Retrival_Ext_QnA_Complete.ipynb"

THIS_NOTEBOOK_NAME = "02_implementation.ipynb"


def select_code_file(file_name: str) -> str:
    """
    Resolve and validate the notebook that will be ingested.

    Parameters
    ----------
    file_name : str
        Name or path of the .ipynb file to ingest.

    Returns
    -------
    str
        Validated path to the code file.
    """

    path = Path(file_name)

    if not path.exists():
        raise FileNotFoundError(
            f"Code file not found: {path}"
        )

    if not path.is_file():
        raise ValueError(
            f"Expected a file, but this is a directory: {path}"
        )

    if path.suffix != ".ipynb":
        raise ValueError(
            f"Expected a .ipynb file, got: {path.suffix!r}"
        )

    if path.name == THIS_NOTEBOOK_NAME:
        raise ValueError(
            "Refusing to ingest this notebook into itself."
        )

    return str(path)


code_file_path = select_code_file(CODE_FILE_NAME)

print("Selected code file:")
print(code_file_path)


def extract_text_from_ipynb(notebook_path: str) -> List[Dict[str, Any]]:
    """
    Extract text from a Jupyter Notebook cell by cell.

    Parameters
    ----------
    notebook_path : str
        Path to the .ipynb file.

    Returns
    -------
    cells_data : List[Dict[str, Any]]
        A list where each element contains cell-wise extracted text
        and metadata.
    """

    cells_data = []

    # Read the Jupyter Notebook
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = nbformat.read(f, as_version=4)

    total_cells = len(notebook.cells)

    print(f"Total cells found: {total_cells}")

    for cell_index, cell in enumerate(notebook.cells):

        # Cell type: markdown, code, or raw
        cell_type = cell.cell_type

        # Extract cell source
        text = cell.source

        # Basic metadata
        cell_number = cell_index + 1
        word_count = len(text.split())
        character_count = len(text)

        cell_info = {
            "cell_number": cell_number,
            "cell_type": cell_type,
            "text": text,
            "word_count": word_count,
            "character_count": character_count
        }

        cells_data.append(cell_info)

    return cells_data


# Extract notebook content
notebook_data = extract_text_from_ipynb(code_file_path)

# Display a sample cell
print(notebook_data[1])

Selected code file:
Retrival_Ext_QnA_Complete.ipynb
Total cells found: 223
{'cell_number': 2, 'cell_type': 'markdown', 'text': '## How This Notebook Will Be Taught\n\nThis notebook has two different pacing styles.\n\n### Part A: Fast Rebuild Mode\n\nFor the parts that students already know from the previous notebook, we will move quickly.\n\nThese include:\n\n- Library installation\n- Imports\n- PDF upload\n- PDF reading\n- Page-wise text extraction\n- Chunk creation\n- Basic BERT-style extractive QnA reader\n- Brute-force QnA recap\n\nFor these repeated sections, we will mostly use:\n\n```text\nSection Heading → Code Cell\n```\nWe will avoid long explanations, concept checks, and detailed observation cells in this part.\n\nThe purpose is only to recreate the required working components in the fresh notebook.\n\n### Part B: Deep Teaching Mode\n\nOnce we reach the new ideas, we will slow down and teach properly.\n\nThese include:\n```text\nWhy brute-force QnA is not scalable\nRetriever 

In [8]:
notebook_data = extract_text_from_ipynb(code_file_path)

print(notebook_data[1])

Total cells found: 223
{'cell_number': 2, 'cell_type': 'markdown', 'text': '## How This Notebook Will Be Taught\n\nThis notebook has two different pacing styles.\n\n### Part A: Fast Rebuild Mode\n\nFor the parts that students already know from the previous notebook, we will move quickly.\n\nThese include:\n\n- Library installation\n- Imports\n- PDF upload\n- PDF reading\n- Page-wise text extraction\n- Chunk creation\n- Basic BERT-style extractive QnA reader\n- Brute-force QnA recap\n\nFor these repeated sections, we will mostly use:\n\n```text\nSection Heading → Code Cell\n```\nWe will avoid long explanations, concept checks, and detailed observation cells in this part.\n\nThe purpose is only to recreate the required working components in the fresh notebook.\n\n### Part B: Deep Teaching Mode\n\nOnce we reach the new ideas, we will slow down and teach properly.\n\nThese include:\n```text\nWhy brute-force QnA is not scalable\nRetriever vs Reader architecture\nText embeddings\nSemantic se

In [9]:
first_cell = notebook_data[1]

print("Cell number:", first_cell["cell_number"])
print("Cell type:", first_cell["cell_type"])
print("Word count:", first_cell["word_count"])
print("Character count:", first_cell["character_count"])
print("\nCell content:")
print(first_cell["text"])

Cell number: 2
Cell type: markdown
Word count: 174
Character count: 1107

Cell content:
## How This Notebook Will Be Taught

This notebook has two different pacing styles.

### Part A: Fast Rebuild Mode

For the parts that students already know from the previous notebook, we will move quickly.

These include:

- Library installation
- Imports
- PDF upload
- PDF reading
- Page-wise text extraction
- Chunk creation
- Basic BERT-style extractive QnA reader
- Brute-force QnA recap

For these repeated sections, we will mostly use:

```text
Section Heading → Code Cell
```
We will avoid long explanations, concept checks, and detailed observation cells in this part.

The purpose is only to recreate the required working components in the fresh notebook.

### Part B: Deep Teaching Mode

Once we reach the new ideas, we will slow down and teach properly.

These include:
```text
Why brute-force QnA is not scalable
Retriever vs Reader architecture
Text embeddings
Semantic search
Cosine similarity
To

In [10]:
for cell in notebook_data:
  print(
      f"Cell {cell['cell_number']:>3} | "
      f"Words: {cell['word_count']:>5} |"
      f"Characters: {cell['character_count']:>6}"
  )


Cell   1 | Words:   146 |Characters:    984
Cell   2 | Words:   174 |Characters:   1107
Cell   3 | Words:    90 |Characters:    545
Cell   4 | Words:   105 |Characters:    687
Cell   5 | Words:   146 |Characters:    959
Cell   6 | Words:   419 |Characters:   2686
Cell   7 | Words:    64 |Characters:    382
Cell   8 | Words:    43 |Characters:    259
Cell   9 | Words:    35 |Characters:    267
Cell  10 | Words:    76 |Characters:    518
Cell  11 | Words:    24 |Characters:    225
Cell  12 | Words:    19 |Characters:    262
Cell  13 | Words:    33 |Characters:    198
Cell  14 | Words:    37 |Characters:    213
Cell  15 | Words:    16 |Characters:    133
Cell  16 | Words:    14 |Characters:     95
Cell  17 | Words:    75 |Characters:    800
Cell  18 | Words:    27 |Characters:    294
Cell  19 | Words:    26 |Characters:    292
Cell  20 | Words:    16 |Characters:    137
Cell  21 | Words:    55 |Characters:    323
Cell  22 | Words:    13 |Characters:     83
Cell  23 | Words:    73 |Charact

In [11]:
low_text_cells = []

for cell in notebook_data:
  if cell['word_count'] < 20:
    low_text_cells.append(cell)

print('cells with fewer than 20 words:', len(low_text_cells))

for cell in low_text_cells:
  print(
      f"Cell {cell['cell_number']} | "
      f"Words: {cell['word_count']} | "
      f"Characters: {cell['character_count']}"
  )

cells with fewer than 20 words: 22
Cell 12 | Words: 19 | Characters: 262
Cell 15 | Words: 16 | Characters: 133
Cell 16 | Words: 14 | Characters: 95
Cell 20 | Words: 16 | Characters: 137
Cell 22 | Words: 13 | Characters: 83
Cell 28 | Words: 14 | Characters: 185
Cell 29 | Words: 12 | Characters: 108
Cell 30 | Words: 19 | Characters: 273
Cell 33 | Words: 16 | Characters: 107
Cell 37 | Words: 15 | Characters: 167
Cell 63 | Words: 11 | Characters: 133
Cell 78 | Words: 19 | Characters: 215
Cell 105 | Words: 18 | Characters: 144
Cell 113 | Words: 17 | Characters: 283
Cell 142 | Words: 15 | Characters: 97
Cell 151 | Words: 18 | Characters: 220
Cell 152 | Words: 10 | Characters: 117
Cell 154 | Words: 14 | Characters: 143
Cell 158 | Words: 6 | Characters: 119
Cell 168 | Words: 6 | Characters: 117
Cell 187 | Words: 19 | Characters: 330
Cell 223 | Words: 0 | Characters: 0


In [12]:
cell_preview_data = []

for cell in notebook_data:
  preview = cell['text'][:120].replace('\n', ' ')
  cell_preview_data.append({
      'cell_number': cell['cell_number'],
      "cell_type": cell["cell_type"],
      'word_count': cell['word_count'],
      'character_count': cell['character_count'],
      'preview': preview
  })

cell_preview_data[:5]

[{'cell_number': 1,
  'cell_type': 'markdown',
  'word_count': 146,
  'character_count': 984,
  'preview': '# From Extractive PDF QnA to Retrieval-Based Document QnA using Embeddings, FAISS, and BERT Reader  In the previous note'},
 {'cell_number': 2,
  'cell_type': 'markdown',
  'word_count': 174,
  'character_count': 1107,
  'preview': '## How This Notebook Will Be Taught  This notebook has two different pacing styles.  ### Part A: Fast Rebuild Mode  For '},
 {'cell_number': 3,
  'cell_type': 'markdown',
  'word_count': 90,
  'character_count': 545,
  'preview': '## Main Story of This Masterclass  The previous system worked like this:  ```text Question → Every Chunk → BERT Reader →'},
 {'cell_number': 4,
  'cell_type': 'markdown',
  'word_count': 105,
  'character_count': 687,
  'preview': '## Important Distinctions  Throughout this notebook, we will keep these ideas separate:  ### Retriever  The retriever fi'},
 {'cell_number': 5,
  'cell_type': 'markdown',
  'word_count': 146,
  'c

In [13]:
# FIX: this cell used to call clean_text() before it was defined
# (clean_text is defined further below), so a clean top-to-bottom run
# died here with NameError: name 'clean_text' is not defined.
#
# The cleaning pass is applied properly after clean_text exists, so
# this cell is intentionally a no-op now.

print("Skipping early cleaning pass: clean_text() is defined below.")

Skipping early cleaning pass: clean_text() is defined below.


In [14]:
print(notebook_data[0]["text"])

# From Extractive PDF QnA to Retrieval-Based Document QnA using Embeddings, FAISS, and BERT Reader

In the previous notebook, we already built a complete **Extractive PDF QnA system** using a BERT-style reader model.

That system covered:

- PDF reading
- Page-wise text extraction
- Text chunking with overlap
- Manual extractive QnA using `start_logits` and `end_logits`
- Answer span extraction
- Chunk-wise QnA
- Answer ranking
- Source tracking
- Failure case analysis

In this new notebook, we will not spend too much time reteaching those parts.

We will quickly rebuild the required pipeline because this is a fresh Colab notebook.

The main focus of this notebook is to move from:

$$
\text{Brute-force Extractive PDF QnA}
$$

to

$$
\text{Retrieval-Based Document QnA}
$$

The final goal is to build this architecture:

$$
\text{Question}
\rightarrow
\text{Retriever}
\rightarrow
\text{Top-k Relevant Chunks}
\rightarrow
\text{BERT Reader}
\rightarrow
\text{Final Answer}
$$


In [15]:
def extract_text_from_ipynb(notebook_path: str) -> List[Dict[str, Any]]:
    """
    Extract text from a Jupyter Notebook cell by cell with metadata.

    Parameters
    ----------
    notebook_path : str
        Path to the .ipynb file.

    Returns
    -------
    cells_data : List[Dict[str, Any]]
        Cell-wise extracted text and metadata.
    """

    cells_data = []

    # Read the notebook
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = nbformat.read(f, as_version=4)

    # Get only the notebook filename
    source_name = Path(notebook_path).name

    total_cells = len(notebook.cells)
    print(f"Total cells found: {total_cells}")

    for cell_index, cell in enumerate(notebook.cells):

        text = cell.source
        cell_type = cell.cell_type

        cell_info = {
            "source": source_name,          # e.g. document_qna.ipynb
            "source_type": "ipynb",         # file type
            "cell_number": cell_index + 1,
            "cell_type": cell_type,         # markdown / code / raw
            "text": text,
            "word_count": len(text.split()),
            "character_count": len(text)
        }

        cells_data.append(cell_info)

    return cells_data

In [16]:
notebook_data = extract_text_from_ipynb(code_file_path)

print(notebook_data[0])

Total cells found: 223
{'source': 'Retrival_Ext_QnA_Complete.ipynb', 'source_type': 'ipynb', 'cell_number': 1, 'cell_type': 'markdown', 'text': '# From Extractive PDF QnA to Retrieval-Based Document QnA using Embeddings, FAISS, and BERT Reader\n\nIn the previous notebook, we already built a complete **Extractive PDF QnA system** using a BERT-style reader model.\n\nThat system covered:\n\n- PDF reading\n- Page-wise text extraction\n- Text chunking with overlap\n- Manual extractive QnA using `start_logits` and `end_logits`\n- Answer span extraction\n- Chunk-wise QnA\n- Answer ranking\n- Source tracking\n- Failure case analysis\n\nIn this new notebook, we will not spend too much time reteaching those parts.\n\nWe will quickly rebuild the required pipeline because this is a fresh Colab notebook.\n\nThe main focus of this notebook is to move from:\n\n$$\n\\text{Brute-force Extractive PDF QnA}\n$$\n\nto\n\n$$\n\\text{Retrieval-Based Document QnA}\n$$\n\nThe final goal is to build this archit

In [17]:
def clean_text(text: str) -> str:
    """
    Clean extracted Jupyter Notebook text for better chunking,
    retrieval, and readability.

    This function:
    1. Removes common tokenizer-style special tokens.
    2. Removes invisible Unicode characters.
    3. Normalizes line endings.
    4. Fixes broken words caused by line wrapping.
    5. Normalizes excessive whitespace.
    6. Preserves meaningful Markdown, mathematical, and Python syntax.

    Parameters
    ----------
    text : str
        Raw text extracted from a Jupyter Notebook cell.

    Returns
    -------
    cleaned : str
        Cleaned notebook text.
    """

    if text is None:
        return ""

    cleaned = str(text)

    # ------------------------------------------------------------
    # 1. Remove common tokenizer/model special tokens
    # ------------------------------------------------------------
    special_tokens = [
        "[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]",
        "<s>", "</s>", "<pad>", "</pad>", "<unk>", "<mask>",
        "<bos>", "</bos>", "<eos>", "</eos>"
    ]

    for token in special_tokens:
        cleaned = cleaned.replace(token, " ")

    # ------------------------------------------------------------
    # 2. Remove invisible Unicode characters
    # ------------------------------------------------------------
    invisible_chars = [
        "\u200b",  # zero-width space
        "\u200c",  # zero-width non-joiner
        "\u200d",  # zero-width joiner
        "\ufeff",  # byte order mark
        "\xa0"     # non-breaking space
    ]

    for char in invisible_chars:
        cleaned = cleaned.replace(char, " ")

    # ------------------------------------------------------------
    # 3. Normalize line endings
    # ------------------------------------------------------------
    cleaned = cleaned.replace("\r\n", "\n")
    cleaned = cleaned.replace("\r", "\n")

    # ------------------------------------------------------------
    # 4. Fix words broken across lines
    # Example:
    # "trans-\nformer" → "transformer"
    # ------------------------------------------------------------
    cleaned = re.sub(r"(\w)-\n(\w)", r"\1\2", cleaned)

    # ------------------------------------------------------------
    # 5. Normalize excessive blank lines
    # ------------------------------------------------------------
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

    # ------------------------------------------------------------
    # 6. Remove trailing spaces from each line
    # ------------------------------------------------------------
    cleaned = "\n".join(
        line.rstrip()
        for line in cleaned.split("\n")
    )

    # ------------------------------------------------------------
    # 7. Normalize repeated spaces
    #
    # IMPORTANT:
    # Do not replace all newlines with spaces.
    # Markdown headings, lists, and Python code depend on structure.
    # ------------------------------------------------------------
    cleaned = re.sub(r"[ \t]+", " ", cleaned)

    # ------------------------------------------------------------
    # 8. Remove unnecessary spaces before punctuation
    # ------------------------------------------------------------
    cleaned = re.sub(r"[ \t]+([.,;:!?])", r"\1", cleaned)

    # ------------------------------------------------------------
    # 9. Remove spaces immediately inside brackets
    # ------------------------------------------------------------
    cleaned = re.sub(r"([\(\[\{])[ \t]+", r"\1", cleaned)
    cleaned = re.sub(r"[ \t]+([\)\]\}])", r"\1", cleaned)

    # ------------------------------------------------------------
    # 10. Final cleanup
    # ------------------------------------------------------------
    cleaned = cleaned.strip()

    return cleaned

In [18]:
noisy_sample = """
[CLS]   ## Introduction to   Transformers [SEP]

Transformers are a type of deep-learning model\u200b that use
self-attention mechanisms to process sequential data.
The byte\u2011pair encoding\u00a0method is commonly used in NLP.

The model consists of several components :
   1. Multi-Head Attention
   2. Feed-Forward Networks
   3. Layer Normalization

The architecture can be represented as :

$$
Attention(Q,K,V) = softmax(\\frac{QK^T}{\\sqrt{d_k}})V
$$

Example code :

    model = Transformer(
        d_model = 512 ,
        n_heads = 8
    )

The word trans-
former is broken across a line.

[PAD] [UNK] <s> </s> <bos> </eos>

Some unnecessary    spaces      appear       here.

There are also spaces before punctuation !   This is wrong .
And brackets are written like ( something ) and [ something ].

Question :   Why does self-attention work ?

Answer : It allows each token to attend to other tokens in the
sequence, capturing contextual relationships.

<pad> <mask> [MASK]

Finally,    this    is    the    end    of    the    document.
"""

print(noisy_sample)


[CLS]   ## Introduction to   Transformers [SEP]

Transformers are a type of deep-learning model​ that use
self-attention mechanisms to process sequential data.
The byte‑pair encoding method is commonly used in NLP.

The model consists of several components :
   1. Multi-Head Attention
   2. Feed-Forward Networks
   3. Layer Normalization

The architecture can be represented as :

$$
Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V
$$

Example code :

    model = Transformer(
        d_model = 512 ,
        n_heads = 8
    )

The word trans-
former is broken across a line.

[PAD] [UNK] <s> </s> <bos> </eos>

Some unnecessary    spaces      appear       here.

There are also spaces before punctuation !   This is wrong .
And brackets are written like ( something ) and [ something ].

Question :   Why does self-attention work ?

Answer : It allows each token to attend to other tokens in the
sequence, capturing contextual relationships.

<pad> <mask> [MASK]

Finally,    this    is    t

In [19]:
cleaned_sample = clean_text(noisy_sample)

print(cleaned_sample)

## Introduction to Transformers

Transformers are a type of deep-learning model that use
self-attention mechanisms to process sequential data.
The byte‑pair encoding method is commonly used in NLP.

The model consists of several components:
 1. Multi-Head Attention
 2. Feed-Forward Networks
 3. Layer Normalization

The architecture can be represented as:

$$
Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V
$$

Example code:

 model = Transformer(
 d_model = 512,
 n_heads = 8
)

The word transformer is broken across a line.



Some unnecessary spaces appear here.

There are also spaces before punctuation! This is wrong.
And brackets are written like (something) and [something].

Question: Why does self-attention work?

Answer: It allows each token to attend to other tokens in the
sequence, capturing contextual relationships.



Finally, this is the end of the document.


In [20]:
def clean_text(text: str, cell_type: str = "markdown") -> str:
    """
    Clean Jupyter Notebook cell text while preserving
    important structure such as Python indentation.

    Parameters
    ----------
    text : str
        Raw text extracted from a notebook cell.

    cell_type : str, default="markdown"
        Type of notebook cell: "markdown", "code", or "raw".

    Returns
    -------
    cleaned : str
        Cleaned text.
    """

    if text is None:
        return ""

    cleaned = str(text)

    # ------------------------------------------------------------
    # 1. Remove common tokenizer/model special tokens
    # ------------------------------------------------------------
    special_tokens = [
        "[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]",
        "<s>", "</s>", "<pad>", "</pad>", "<unk>", "<mask>",
        "<bos>", "</bos>", "<eos>", "</eos>"
    ]

    for token in special_tokens:
        cleaned = cleaned.replace(token, " ")

    # ------------------------------------------------------------
    # 2. Remove invisible Unicode characters
    # ------------------------------------------------------------
    invisible_chars = [
        "\u200b",  # zero-width space
        "\u200c",  # zero-width non-joiner
        "\u200d",  # zero-width joiner
        "\ufeff",  # byte order mark
        "\xa0"     # non-breaking space
    ]

    for char in invisible_chars:
        cleaned = cleaned.replace(char, " ")

    # ------------------------------------------------------------
    # 3. Normalize line endings
    # ------------------------------------------------------------
    cleaned = cleaned.replace("\r\n", "\n")
    cleaned = cleaned.replace("\r", "\n")

    # ------------------------------------------------------------
    # 4. Fix words broken across lines
    # Example:
    # trans-
    # former
    #
    # becomes:
    # transformer
    # ------------------------------------------------------------
    cleaned = re.sub(r"(\w)-\n(\w)", r"\1\2", cleaned)

    # ============================================================
    # CODE CELL CLEANING
    # ============================================================
    if cell_type == "code":

        # Remove trailing whitespace while preserving indentation
        lines = cleaned.split("\n")

        cleaned_lines = []

        for line in lines:
            # Remove trailing spaces only
            line = line.rstrip()
            cleaned_lines.append(line)

        cleaned = "\n".join(cleaned_lines)

        # Remove excessive blank lines
        cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

        # Do NOT normalize spaces/tabs inside code.
        # Python indentation must remain intact.

    # ============================================================
    # MARKDOWN / RAW CELL CLEANING
    # ============================================================
    else:

        # Remove excessive blank lines
        cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

        # Remove trailing whitespace from each line
        cleaned = "\n".join(
            line.rstrip()
            for line in cleaned.split("\n")
        )

        # Normalize repeated spaces/tabs in prose
        cleaned = re.sub(r"[ \t]+", " ", cleaned)

        # Fix spacing before punctuation
        cleaned = re.sub(
            r"[ \t]+([.,;:!?])",
            r"\1",
            cleaned
        )

        # Remove unnecessary spaces inside brackets
        cleaned = re.sub(
            r"([\(\[\{])[ \t]+",
            r"\1",
            cleaned
        )

        cleaned = re.sub(
            r"[ \t]+([\)\]\}])",
            r"\1",
            cleaned
        )

    # ------------------------------------------------------------
    # 5. Final cleanup
    # ------------------------------------------------------------
    cleaned = cleaned.strip()

    return cleaned

In [21]:
for cell in notebook_data:
    cell["text"] = clean_text(
        cell["text"],
        cell["cell_type"]
    )

In [22]:
noisy_code = """[CLS]
def train_model(model, X_train, y_train):
    model.fit(
        X_train,
        y_train
    )

    if model.score(X_train, y_train) > 0.90:
        print("Excellent model")

    return model
[SEP]
"""

print("BEFORE CLEANING:")
print(noisy_code)

print("\nAFTER CLEANING:")
print(clean_text(noisy_code, "code"))

BEFORE CLEANING:
[CLS]
def train_model(model, X_train, y_train):
    model.fit(
        X_train,
        y_train
    )

    if model.score(X_train, y_train) > 0.90:
        print("Excellent model")

    return model
[SEP]


AFTER CLEANING:
def train_model(model, X_train, y_train):
    model.fit(
        X_train,
        y_train
    )

    if model.score(X_train, y_train) > 0.90:
        print("Excellent model")

    return model


In [23]:
messy_code = """[CLS]
def train_model(model, X_train, y_train):

    model.fit(X_train, y_train)

    if model.score(X_train, y_train) > 0.90:
        print("Excellent model")

        if True:
            print("Nested block")

    return model

[SEP]
"""

cleaned_code = clean_text(messy_code, "code")

print("BEFORE:")
print(repr(messy_code))

print("\nAFTER:")
print(repr(cleaned_code))

BEFORE:
'[CLS]\ndef train_model(model, X_train, y_train):\n\n    model.fit(X_train, y_train)\n\n    if model.score(X_train, y_train) > 0.90:\n        print("Excellent model")\n\n        if True:\n            print("Nested block")\n\n    return model\n\n[SEP]\n'

AFTER:
'def train_model(model, X_train, y_train):\n\n    model.fit(X_train, y_train)\n\n    if model.score(X_train, y_train) > 0.90:\n        print("Excellent model")\n\n        if True:\n            print("Nested block")\n\n    return model'


In [24]:
print("Indentation preserved:",
      "\n    model.fit" in cleaned_code)

Indentation preserved: True


In [25]:
cleaned_cells_data = []

for cell in notebook_data:

    raw_text = cell["text"]

    cleaned_cell_text = clean_text(
        raw_text,
        cell["cell_type"]
    )

    cleaned_cell_info = {

        "source": cell["source"],
        "source_type": cell["source_type"],

        "cell_number": cell["cell_number"],
        "cell_type": cell["cell_type"],

        "raw_text": raw_text,
        "cleaned_text": cleaned_cell_text,

        "raw_word_count": cell["word_count"],
        "cleaned_word_count": len(cleaned_cell_text.split()),

        "raw_character_count": cell["character_count"],
        "cleaned_character_count": len(cleaned_cell_text)

    }

    cleaned_cells_data.append(cleaned_cell_info)


print("Text cleaning completed.")
print("Total cells cleaned:", len(cleaned_cells_data))

Text cleaning completed.
Total cells cleaned: 223


In [26]:
first_cleaned_cell = cleaned_cells_data[0]

print("Source:", first_cleaned_cell["source"])
print("Source type:", first_cleaned_cell["source_type"])
print("Cell number:", first_cleaned_cell["cell_number"])
print("Cell type:", first_cleaned_cell["cell_type"])

print("\nRaw text:")
print(first_cleaned_cell["raw_text"])

print("\nCleaned text:")
print(first_cleaned_cell["cleaned_text"])

Source: Retrival_Ext_QnA_Complete.ipynb
Source type: ipynb
Cell number: 1
Cell type: markdown

Raw text:
# From Extractive PDF QnA to Retrieval-Based Document QnA using Embeddings, FAISS, and BERT Reader

In the previous notebook, we already built a complete **Extractive PDF QnA system** using a BERT-style reader model.

That system covered:

- PDF reading
- Page-wise text extraction
- Text chunking with overlap
- Manual extractive QnA using `start_logits` and `end_logits`
- Answer span extraction
- Chunk-wise QnA
- Answer ranking
- Source tracking
- Failure case analysis

In this new notebook, we will not spend too much time reteaching those parts.

We will quickly rebuild the required pipeline because this is a fresh Colab notebook.

The main focus of this notebook is to move from:

$$
\text{Brute-force Extractive PDF QnA}
$$

to

$$
\text{Retrieval-Based Document QnA}
$$

The final goal is to build this architecture:

$$
\text{Question}
\rightarrow
\text{Retriever}
\rightarrow
\text

In [27]:
print("Raw word count:", first_cleaned_cell["raw_word_count"])
print("Cleaned word count:", first_cleaned_cell["cleaned_word_count"])

print("Raw character count:", first_cleaned_cell["raw_character_count"])
print("Cleaned character count:", first_cleaned_cell["cleaned_character_count"])


Raw word count: 146
Cleaned word count: 146
Raw character count: 984
Cleaned character count: 984


In [28]:
for cell in cleaned_cells_data:

    if cell["cell_type"] == "code":

        print("=" * 80)
        print(f"Source: {cell['source']}")
        print(f"Cell number: {cell['cell_number']}")
        print("\nCleaned code:\n")
        print(cell["cleaned_text"])

        break

Source: Retrival_Ext_QnA_Complete.ipynb
Cell number: 9

Cleaned code:

# Install required libraries
# PyMuPDF -> PDF reading
# transformers -> BERT-style QnA reader
# sentence-transformers -> embedding model
# faiss-cpu -> efficient vector similarity search

!pip install -q pymupdf transformers accelerate sentence-transformers faiss-cpu


In [29]:
# FIX: `code_cell` was used here without ever being defined,
# which raised NameError. Define it first.

code_cell = next(
    cell for cell in cleaned_cells_data
    if cell["cell_type"] == "code"
)

print("RAW CODE:")
print(code_cell["raw_text"])

print("\n" + "=" * 80 + "\n")

print("CLEANED CODE:")
print(code_cell["cleaned_text"])

RAW CODE:
# Install required libraries
# PyMuPDF -> PDF reading
# transformers -> BERT-style QnA reader
# sentence-transformers -> embedding model
# faiss-cpu -> efficient vector similarity search

!pip install -q pymupdf transformers accelerate sentence-transformers faiss-cpu


CLEANED CODE:
# Install required libraries
# PyMuPDF -> PDF reading
# transformers -> BERT-style QnA reader
# sentence-transformers -> embedding model
# faiss-cpu -> efficient vector similarity search

!pip install -q pymupdf transformers accelerate sentence-transformers faiss-cpu


In [30]:
cleaned_cells_data = []

for cell in notebook_data:

    raw_text = cell["text"]

    cleaned_cell_text = clean_text(
        raw_text,
        cell["cell_type"]
    )

    cleaned_cell_info = {
        "source": cell["source"],
        "source_type": cell["source_type"],
        "cell_number": cell["cell_number"],
        "cell_type": cell["cell_type"],
        "raw_text": raw_text,
        "cleaned_text": cleaned_cell_text,
        "raw_word_count": cell["word_count"],
        "cleaned_word_count": len(cleaned_cell_text.split()),
        "raw_character_count": cell["character_count"],
        "cleaned_character_count": len(cleaned_cell_text)
    }

    cleaned_cells_data.append(cleaned_cell_info)

print("Text cleaning completed.")
print("Total cells cleaned:", len(cleaned_cells_data))

Text cleaning completed.
Total cells cleaned: 223


In [31]:
# Display word and character count before and after cleaning

for cell in cleaned_cells_data:
    print(
        f"Cell {cell['cell_number']:>3} | "
        f"Raw words: {cell['raw_word_count']:>5} | "
        f"Cleaned words: {cell['cleaned_word_count']:>5} | "
        f"Raw chars: {cell['raw_character_count']:>6} | "
        f"Cleaned chars: {cell['cleaned_character_count']:>6}"
    )

Cell   1 | Raw words:   146 | Cleaned words:   146 | Raw chars:    984 | Cleaned chars:    984
Cell   2 | Raw words:   174 | Cleaned words:   174 | Raw chars:   1107 | Cleaned chars:   1107
Cell   3 | Raw words:    90 | Cleaned words:    90 | Raw chars:    545 | Cleaned chars:    545
Cell   4 | Raw words:   105 | Cleaned words:   105 | Raw chars:    687 | Cleaned chars:    687
Cell   5 | Raw words:   146 | Cleaned words:   146 | Raw chars:    959 | Cleaned chars:    949
Cell   6 | Raw words:   419 | Cleaned words:   419 | Raw chars:   2686 | Cleaned chars:   2650
Cell   7 | Raw words:    64 | Cleaned words:    64 | Raw chars:    382 | Cleaned chars:    382
Cell   8 | Raw words:    43 | Cleaned words:    43 | Raw chars:    259 | Cleaned chars:    259
Cell   9 | Raw words:    35 | Cleaned words:    35 | Raw chars:    267 | Cleaned chars:    267
Cell  10 | Raw words:    76 | Cleaned words:    76 | Raw chars:    518 | Cleaned chars:    518
Cell  11 | Raw words:    24 | Cleaned words:    24

In [32]:
# Inspect the cell where cleaned word count increased

cell_36 = next(
    cell for cell in cleaned_cells_data
    if cell["cell_number"] == 36
)

print("RAW TEXT:")
print(cell_36["raw_text"])

print("\n" + "=" * 80 + "\n")

print("CLEANED TEXT:")
print(cell_36["cleaned_text"])

print("\n" + "=" * 80 + "\n")

print("Raw word count:", cell_36["raw_word_count"])
print("Cleaned word count:", cell_36["cleaned_word_count"])

RAW TEXT:
def clean_answer_for_display(answer: str) -> str:
    """
    Cleans final extracted answer for readable display.

    Handles:
    - Special tokens
    - WordPiece artifacts
    - Extra spaces
    - Broken punctuation spacing
    """

    if answer is None:
        return ""

    answer = str(answer)

    # Remove common special tokens if they appear accidentally
    special_tokens = [
        " ", " ", " ", " ", " ",
        " ", " ", " ", " "
    ]

    for token in special_tokens:
        answer = answer.replace(token, "")

    # Remove WordPiece continuation markers
    # Example: "transform ##er" -> "transformer"
    answer = answer.replace(" ##", "")
    answer = answer.replace("##", "")

    # Unicode normalization
    answer = unicodedata.normalize("NFKC", answer)

    # Remove repeated spaces
    answer = re.sub(r"\s+", " ", answer)

    # Fix spaces before punctuation
    answer = re.sub(r"\s+([.,;:!?%)\]])", r"\1", answer)

    # Fix spaces after opening brackets


In [33]:
raw = cell_36["raw_text"]
cleaned = cell_36["cleaned_text"]

print("Raw words:", len(raw.split()))
print("Cleaned words:", len(cleaned.split()))

print("\nCharacters changed:")

for i, (r, c) in enumerate(zip(raw, cleaned)):
    if r != c:
        print(
            f"Position {i}: "
            f"raw={repr(r)} → cleaned={repr(c)}"
        )

Raw words: 149
Cleaned words: 149

Characters changed:


In [34]:
print("Invisible characters found:")

for i, char in enumerate(raw):
    if char in ["\u200b", "\u200c", "\u200d", "\ufeff", "\xa0"]:
        print(
            f"Position {i}: "
            f"{repr(char)}"
        )

Invisible characters found:


In [35]:
cleaned_cells_data = []

for cell in notebook_data:

    raw_text = cell["text"]

    cleaned_cell_text = clean_text(
        raw_text,
        cell["cell_type"]
    )

    cleaned_cell_info = {
        "source": cell["source"],
        "source_type": cell["source_type"],
        "cell_number": cell["cell_number"],
        "cell_type": cell["cell_type"],

        "raw_text": raw_text,
        "cleaned_text": cleaned_cell_text,

        "raw_word_count": cell["word_count"],
        "cleaned_word_count": len(cleaned_cell_text.split()),

        "raw_character_count": cell["character_count"],
        "cleaned_character_count": len(cleaned_cell_text)
    }

    cleaned_cells_data.append(cleaned_cell_info)

print("Text cleaning completed.")
print("Total cells cleaned:", len(cleaned_cells_data))

Text cleaning completed.
Total cells cleaned: 223


In [36]:
for cell in cleaned_cells_data:
    print(
        f"Cell {cell['cell_number']:>3} | "
        f"Raw words: {cell['raw_word_count']:>5} | "
        f"Cleaned words: {cell['cleaned_word_count']:>5} | "
        f"Raw chars: {cell['raw_character_count']:>6} | "
        f"Cleaned chars: {cell['cleaned_character_count']:>6}"
    )

Cell   1 | Raw words:   146 | Cleaned words:   146 | Raw chars:    984 | Cleaned chars:    984
Cell   2 | Raw words:   174 | Cleaned words:   174 | Raw chars:   1107 | Cleaned chars:   1107
Cell   3 | Raw words:    90 | Cleaned words:    90 | Raw chars:    545 | Cleaned chars:    545
Cell   4 | Raw words:   105 | Cleaned words:   105 | Raw chars:    687 | Cleaned chars:    687
Cell   5 | Raw words:   146 | Cleaned words:   146 | Raw chars:    959 | Cleaned chars:    949
Cell   6 | Raw words:   419 | Cleaned words:   419 | Raw chars:   2686 | Cleaned chars:   2650
Cell   7 | Raw words:    64 | Cleaned words:    64 | Raw chars:    382 | Cleaned chars:    382
Cell   8 | Raw words:    43 | Cleaned words:    43 | Raw chars:    259 | Cleaned chars:    259
Cell   9 | Raw words:    35 | Cleaned words:    35 | Raw chars:    267 | Cleaned chars:    267
Cell  10 | Raw words:    76 | Cleaned words:    76 | Raw chars:    518 | Cleaned chars:    518
Cell  11 | Raw words:    24 | Cleaned words:    24

In [37]:
cell_36 = next(
    cell for cell in cleaned_cells_data
    if cell["cell_number"] == 36
)

print("Cell type:", cell_36["cell_type"])
print("Raw words:", cell_36["raw_word_count"])
print("Cleaned words:", cell_36["cleaned_word_count"])
print("Raw chars:", cell_36["raw_character_count"])
print("Cleaned chars:", cell_36["cleaned_character_count"])

print("\nCLEANED CODE:\n")
print(cell_36["cleaned_text"])

Cell type: code
Raw words: 140
Cleaned words: 149
Raw chars: 1200
Cleaned chars: 1166

CLEANED CODE:

def clean_answer_for_display(answer: str) -> str:
    """
    Cleans final extracted answer for readable display.

    Handles:
    - Special tokens
    - WordPiece artifacts
    - Extra spaces
    - Broken punctuation spacing
    """

    if answer is None:
        return ""

    answer = str(answer)

    # Remove common special tokens if they appear accidentally
    special_tokens = [
        " ", " ", " ", " ", " ",
        " ", " ", " ", " "
    ]

    for token in special_tokens:
        answer = answer.replace(token, "")

    # Remove WordPiece continuation markers
    # Example: "transform ##er" -> "transformer"
    answer = answer.replace(" ##", "")
    answer = answer.replace("##", "")

    # Unicode normalization
    answer = unicodedata.normalize("NFKC", answer)

    # Remove repeated spaces
    answer = re.sub(r"\s+", " ", answer)

    # Fix spaces before punctuation
    an

In [38]:
def clean_text(text: str, cell_type: str = "markdown") -> str:
    """
    Clean Jupyter Notebook cell text while preserving
    Python code structure and Markdown structure.

    Parameters
    ----------
    text : str
        Raw text extracted from a notebook cell.

    cell_type : str, default="markdown"
        Type of notebook cell: "markdown", "code", or "raw".

    Returns
    -------
    cleaned : str
        Cleaned text.
    """

    if text is None:
        return ""

    cleaned = str(text)

    # ============================================================
    # CODE CELL
    # ============================================================
    # Code must be treated conservatively.
    # Do not normalize spaces, punctuation, brackets, or tokens.
    # These may be part of the actual Python source.
    # ============================================================

    if cell_type == "code":

        # Normalize line endings only
        cleaned = cleaned.replace("\r\n", "\n")
        cleaned = cleaned.replace("\r", "\n")

        # Remove trailing whitespace only.
        # Leading whitespace/indentation is preserved.
        lines = cleaned.split("\n")

        cleaned_lines = [
            line.rstrip()
            for line in lines
        ]

        cleaned = "\n".join(cleaned_lines)

        # Remove excessive blank lines
        cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

        return cleaned.strip()

    # ============================================================
    # MARKDOWN / RAW CELL
    # ============================================================

    # ------------------------------------------------------------
    # 0. Protect fenced code blocks.
    #
    # FIX: the prose rules below collapse every run of spaces/tabs.
    # Applied to a ```python block inside a Markdown cell that
    # destroys Python indentation - which is exactly the structure
    # a code-retrieval corpus needs. Stash fenced blocks first and
    # restore them untouched at the end.
    # ------------------------------------------------------------

    fenced_blocks = []

    def _stash_fence(match):
        fenced_blocks.append(match.group(0))
        return f"<FENCEDBLOCK{len(fenced_blocks) - 1}>"

    cleaned = re.sub(
        r"```.*?```",
        _stash_fence,
        cleaned,
        flags=re.DOTALL
    )

    # ------------------------------------------------------------
    # 1. Remove common tokenizer/model special tokens
    # ------------------------------------------------------------
    special_tokens = [
        "[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]",
        "<s>", "</s>", "<pad>", "</pad>", "<unk>", "<mask>",
        "<bos>", "</bos>", "<eos>", "</eos>"
    ]

    for token in special_tokens:
        cleaned = cleaned.replace(token, " ")

    # ------------------------------------------------------------
    # 2. Remove invisible Unicode characters
    # ------------------------------------------------------------
    invisible_chars = [
        "\u200b",
        "\u200c",
        "\u200d",
        "\ufeff",
        "\xa0"
    ]

    for char in invisible_chars:
        cleaned = cleaned.replace(char, " ")

    # ------------------------------------------------------------
    # 3. Normalize line endings
    # ------------------------------------------------------------
    cleaned = cleaned.replace("\r\n", "\n")
    cleaned = cleaned.replace("\r", "\n")

    # ------------------------------------------------------------
    # 4. Fix broken words across lines
    # Example:
    # trans-
    # former
    #
    # becomes:
    # transformer
    # ------------------------------------------------------------
    cleaned = re.sub(r"(\w)-\n(\w)", r"\1\2", cleaned)

    # ------------------------------------------------------------
    # 5. Normalize excessive blank lines
    # ------------------------------------------------------------
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

    # ------------------------------------------------------------
    # 6. Remove trailing whitespace
    # ------------------------------------------------------------
    cleaned = "\n".join(
        line.rstrip()
        for line in cleaned.split("\n")
    )

    # ------------------------------------------------------------
    # 7. Normalize repeated spaces
    # ------------------------------------------------------------
    cleaned = re.sub(r"[ \t]+", " ", cleaned)

    # ------------------------------------------------------------
    # 8. Fix spacing before punctuation
    # ------------------------------------------------------------
    cleaned = re.sub(
        r"[ \t]+([.,;:!?])",
        r"\1",
        cleaned
    )

    # ------------------------------------------------------------
    # 9. Remove unnecessary spaces inside brackets
    # ------------------------------------------------------------
    cleaned = re.sub(
        r"([\(\[\{])[ \t]+",
        r"\1",
        cleaned
    )

    cleaned = re.sub(
        r"[ \t]+([\)\]\}])",
        r"\1",
        cleaned
    )

    # ------------------------------------------------------------
    # 10. Restore fenced code blocks exactly as they were
    # ------------------------------------------------------------
    for fence_index, block in enumerate(fenced_blocks):
        cleaned = cleaned.replace(
            f"<FENCEDBLOCK{fence_index}>",
            block
        )

    # ------------------------------------------------------------
    # 11. Final cleanup
    # ------------------------------------------------------------
    cleaned = cleaned.strip()

    return cleaned

In [39]:
cleaned_cells_data = []

for cell in notebook_data:

    raw_text = cell["text"]

    cleaned_cell_text = clean_text(
        raw_text,
        cell["cell_type"]
    )

    cleaned_cell_info = {
        "source": cell["source"],
        "source_type": cell["source_type"],
        "cell_number": cell["cell_number"],
        "cell_type": cell["cell_type"],

        "raw_text": raw_text,
        "cleaned_text": cleaned_cell_text,

        "raw_word_count": cell["word_count"],
        "cleaned_word_count": len(cleaned_cell_text.split()),

        "raw_character_count": cell["character_count"],
        "cleaned_character_count": len(cleaned_cell_text)
    }

    cleaned_cells_data.append(cleaned_cell_info)

print("Text cleaning completed.")
print("Total cells cleaned:", len(cleaned_cells_data))

Text cleaning completed.
Total cells cleaned: 223


In [40]:
cell_36 = next(
    cell for cell in cleaned_cells_data
    if cell["cell_number"] == 36
)

print("Cell type:", cell_36["cell_type"])
print("Raw words:", cell_36["raw_word_count"])
print("Cleaned words:", cell_36["cleaned_word_count"])
print("Raw chars:", cell_36["raw_character_count"])
print("Cleaned chars:", cell_36["cleaned_character_count"])

print("\nCLEANED CODE:\n")
print(cell_36["cleaned_text"])

Cell type: code
Raw words: 140
Cleaned words: 149
Raw chars: 1200
Cleaned chars: 1166

CLEANED CODE:

def clean_answer_for_display(answer: str) -> str:
    """
    Cleans final extracted answer for readable display.

    Handles:
    - Special tokens
    - WordPiece artifacts
    - Extra spaces
    - Broken punctuation spacing
    """

    if answer is None:
        return ""

    answer = str(answer)

    # Remove common special tokens if they appear accidentally
    special_tokens = [
        " ", " ", " ", " ", " ",
        " ", " ", " ", " "
    ]

    for token in special_tokens:
        answer = answer.replace(token, "")

    # Remove WordPiece continuation markers
    # Example: "transform ##er" -> "transformer"
    answer = answer.replace(" ##", "")
    answer = answer.replace("##", "")

    # Unicode normalization
    answer = unicodedata.normalize("NFKC", answer)

    # Remove repeated spaces
    answer = re.sub(r"\s+", " ", answer)

    # Fix spaces before punctuation
    an

#Change

In [41]:
notebook_data = extract_text_from_ipynb(code_file_path)

print("Notebook re-extracted.")
print("Total cells:", len(notebook_data))

Total cells found: 223
Notebook re-extracted.
Total cells: 223


In [42]:
cell_36_raw = next(
    cell for cell in notebook_data
    if cell["cell_number"] == 36
)

print("Cell type:", cell_36_raw["cell_type"])
print("Word count:", cell_36_raw["word_count"])
print("Character count:", cell_36_raw["character_count"])

print("\nRAW CODE:")
print(cell_36_raw["text"])

Cell type: code
Word count: 140
Character count: 1200

RAW CODE:
def clean_answer_for_display(answer: str) -> str:
    """
    Cleans final extracted answer for readable display.

    Handles:
    - Special tokens
    - WordPiece artifacts
    - Extra spaces
    - Broken punctuation spacing
    """

    if answer is None:
        return ""

    answer = str(answer)

    # Remove common special tokens if they appear accidentally
    special_tokens = [
        "[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]",
        "<s>", "</s>", "<pad>", "<unk>"
    ]

    for token in special_tokens:
        answer = answer.replace(token, "")

    # Remove WordPiece continuation markers
    # Example: "transform ##er" -> "transformer"
    answer = answer.replace(" ##", "")
    answer = answer.replace("##", "")

    # Unicode normalization
    answer = unicodedata.normalize("NFKC", answer)

    # Remove repeated spaces
    answer = re.sub(r"\s+", " ", answer)

    # Fix spaces before punctuation
    answe

In [43]:
cleaned_cells_data = []

for cell in notebook_data:

    raw_text = cell["text"]

    cleaned_cell_text = clean_text(
        raw_text,
        cell["cell_type"]
    )

    cleaned_cell_info = {
        "source": cell["source"],
        "source_type": cell["source_type"],
        "cell_number": cell["cell_number"],
        "cell_type": cell["cell_type"],

        "raw_text": raw_text,
        "cleaned_text": cleaned_cell_text,

        "raw_word_count": cell["word_count"],
        "cleaned_word_count": len(cleaned_cell_text.split()),

        "raw_character_count": cell["character_count"],
        "cleaned_character_count": len(cleaned_cell_text)
    }

    cleaned_cells_data.append(cleaned_cell_info)

print("Text cleaning completed.")
print("Total cells cleaned:", len(cleaned_cells_data))

Text cleaning completed.
Total cells cleaned: 223


In [44]:
cell_36 = next(
    cell for cell in cleaned_cells_data
    if cell["cell_number"] == 36
)

print("Cell type:", cell_36["cell_type"])

print("Raw words:", cell_36["raw_word_count"])
print("Cleaned words:", cell_36["cleaned_word_count"])

print("Raw chars:", cell_36["raw_character_count"])
print("Cleaned chars:", cell_36["cleaned_character_count"])

print("\nRAW CODE:")
print(cell_36["raw_text"])

print("\n" + "=" * 80 + "\n")

print("CLEANED CODE:")
print(cell_36["cleaned_text"])

Cell type: code
Raw words: 140
Cleaned words: 140
Raw chars: 1200
Cleaned chars: 1200

RAW CODE:
def clean_answer_for_display(answer: str) -> str:
    """
    Cleans final extracted answer for readable display.

    Handles:
    - Special tokens
    - WordPiece artifacts
    - Extra spaces
    - Broken punctuation spacing
    """

    if answer is None:
        return ""

    answer = str(answer)

    # Remove common special tokens if they appear accidentally
    special_tokens = [
        "[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]",
        "<s>", "</s>", "<pad>", "<unk>"
    ]

    for token in special_tokens:
        answer = answer.replace(token, "")

    # Remove WordPiece continuation markers
    # Example: "transform ##er" -> "transformer"
    answer = answer.replace(" ##", "")
    answer = answer.replace("##", "")

    # Unicode normalization
    answer = unicodedata.normalize("NFKC", answer)

    # Remove repeated spaces
    answer = re.sub(r"\s+", " ", answer)

    # Fix spa

In [45]:
def split_into_sentences(
    text: str,
    cell_type: str = "markdown"
) -> List[str]:
    """
    Split cleaned Jupyter Notebook cell text into
    sentence-like text units.

    This function is intentionally simple and dependency-free
    for Colab teaching.

    Parameters
    ----------
    text : str
        Cleaned text from a notebook cell.

    cell_type : str, default="markdown"
        Type of notebook cell: "markdown", "code", or "raw".

    Returns
    -------
    sentences : List[str]
        List of sentence-like text units.
    """

    if text is None:
        return []

    text = clean_text(text, cell_type)

    if len(text.strip()) == 0:
        return []

    # ============================================================
    # CODE CELLS
    # ============================================================
    # Python code should not be split using sentence punctuation.
    # A period inside a string, comment, or expression may not
    # represent the end of a sentence.
    # ============================================================

    if cell_type == "code":
        return [text.strip()]

    # ============================================================
    # MARKDOWN / RAW CELLS
    # ============================================================

    # Protect common abbreviations so they do not split incorrectly
    abbreviation_map = {
        "e.g.": "e<DOT>g<DOT>",
        "i.e.": "i<DOT>e<DOT>",
        "Fig.": "Fig<DOT>",
        "fig.": "fig<DOT>",
        "Eq.": "Eq<DOT>",
        "eq.": "eq<DOT>",
        "Dr.": "Dr<DOT>",
        "Mr.": "Mr<DOT>",
        "Ms.": "Ms<DOT>",
        "Prof.": "Prof<DOT>",
        "vs.": "vs<DOT>",
        "et al.": "et al<DOT>"
    }

    protected_text = text

    for original, protected in abbreviation_map.items():
        protected_text = protected_text.replace(
            original,
            protected
        )

    # Split when punctuation is followed by whitespace
    # and a likely new sentence start.
    sentence_candidates = re.split(
        r'(?<=[.!?])\s+(?=[A-Z0-9"\(\[])',
        protected_text
    )

    sentences = []

    for sentence in sentence_candidates:

        # Restore protected dots
        sentence = sentence.replace("<DOT>", ".")

        # Clean while preserving Markdown behavior
        sentence = clean_text(
            sentence,
            cell_type
        )

        if len(sentence.strip()) > 0:
            sentences.append(sentence)

    return sentences

In [46]:
first_cell = notebook_data[0]

sentences = split_into_sentences(
    first_cell["text"],
    first_cell["cell_type"]
)

print("Number of sentences:", len(sentences))

for i, sentence in enumerate(sentences[:5], start=1):
    print(f"{i}. {sentence}")

Number of sentences: 4
1. # From Extractive PDF QnA to Retrieval-Based Document QnA using Embeddings, FAISS, and BERT Reader

In the previous notebook, we already built a complete **Extractive PDF QnA system** using a BERT-style reader model.
2. That system covered:

- PDF reading
- Page-wise text extraction
- Text chunking with overlap
- Manual extractive QnA using `start_logits` and `end_logits`
- Answer span extraction
- Chunk-wise QnA
- Answer ranking
- Source tracking
- Failure case analysis

In this new notebook, we will not spend too much time reteaching those parts.
3. We will quickly rebuild the required pipeline because this is a fresh Colab notebook.
4. The main focus of this notebook is to move from:

$$
\text{Brute-force Extractive PDF QnA}
$$

to

$$
\text{Retrieval-Based Document QnA}
$$

The final goal is to build this architecture:

$$
\text{Question}
\rightarrow
\text{Retriever}
\rightarrow
\text{Top-k Relevant Chunks}
\rightarrow
\text{BERT Reader}
\rightarrow
\text{

In [47]:
code_cell = next(
    cell for cell in notebook_data
    if cell["cell_type"] == "code"
)

code_units = split_into_sentences(
    code_cell["text"],
    code_cell["cell_type"]
)

print("Number of code units:", len(code_units))

print("\nCode:")
print(code_units[0])

Number of code units: 1

Code:
# Install required libraries
# PyMuPDF -> PDF reading
# transformers -> BERT-style QnA reader
# sentence-transformers -> embedding model
# faiss-cpu -> efficient vector similarity search

!pip install -q pymupdf transformers accelerate sentence-transformers faiss-cpu


In [48]:
def chunk_notebook_with_metadata(
    cleaned_cells_data: List[Dict[str, Any]],
    max_words: int = 250,
    sentence_overlap: int = 2
) -> List[Dict[str, Any]]:
    """
    Create structure-aware chunks from cleaned Jupyter Notebook cells.

    Markdown and raw cells are split into sentence-like units.
    Code cells are treated as complete structural units so that
    Python indentation and code relationships are preserved.

    Parameters
    ----------
    cleaned_cells_data : List[Dict[str, Any]]
        List of cell dictionaries containing cleaned text and metadata.

    max_words : int
        Approximate maximum number of words per chunk.

    sentence_overlap : int
        Number of sentence-like units to overlap between consecutive
        chunks within the same cell.

    Returns
    -------
    chunks_data : List[Dict[str, Any]]
        List of chunk dictionaries with source, cell, and chunk metadata.
    """

    if max_words <= 0:
        raise ValueError("max_words must be greater than 0.")

    if sentence_overlap < 0:
        raise ValueError("sentence_overlap cannot be negative.")

    chunks_data = []
    global_chunk_id = 0

    for cell in cleaned_cells_data:

        source = cell["source"]
        source_type = cell["source_type"]
        cell_number = cell["cell_number"]
        cell_type = cell["cell_type"]

        cell_text = cell["cleaned_text"]

        if len(cell_text.strip()) == 0:
            continue

        # --------------------------------------------------------
        # Split the cell into structural units
        # --------------------------------------------------------

        units = split_into_sentences(
            cell_text,
            cell_type
        )

        if len(units) == 0:
            continue

        # --------------------------------------------------------
        # Handle units longer than max_words
        # --------------------------------------------------------

        processed_units = []

        for unit in units:

            words = unit.split()

            if len(words) <= max_words:

                processed_units.append(unit)

            elif cell_type == "code":

                # FIX: the old fallback did `" ".join(words[...])` for
                # every unit, including code. That collapsed the whole
                # code cell onto a single line and destroyed newlines
                # and indentation - 11 of 110 code chunks came out as
                # unreadable one-liners.
                #
                # Split long code on LINE boundaries instead, so the
                # stored chunk is still valid, readable Python.

                piece_lines = []
                piece_words = 0

                for line in unit.split("\n"):

                    line_words = len(line.split())

                    if (
                        piece_lines
                        and piece_words + line_words > max_words
                    ):

                        piece = "\n".join(piece_lines)

                        if len(piece.strip()) > 0:
                            processed_units.append(piece)

                        piece_lines = []
                        piece_words = 0

                    piece_lines.append(line)
                    piece_words += line_words

                piece = "\n".join(piece_lines)

                if len(piece.strip()) > 0:
                    processed_units.append(piece)

            else:

                # Prose fallback for very long Markdown units.
                for start in range(
                    0,
                    len(words),
                    max_words
                ):

                    piece = " ".join(
                        words[start:start + max_words]
                    )

                    piece = clean_text(
                        piece,
                        cell_type
                    )

                    if len(piece.strip()) > 0:
                        processed_units.append(piece)

        # --------------------------------------------------------
        # Build chunks
        # --------------------------------------------------------

        current_units = []
        current_word_count = 0

        for unit in processed_units:

            unit = clean_text(
                unit,
                cell_type
            )

            unit_word_count = len(unit.split())

            if unit_word_count == 0:
                continue

            # ----------------------------------------------------
            # If adding this unit exceeds max_words,
            # finalize the current chunk first.
            # ----------------------------------------------------

            if (
                current_units
                and current_word_count + unit_word_count > max_words
            ):

                chunk_text = "\n".join(
                    current_units
                    if cell_type == "code"
                    else current_units
                )

                chunk_text = clean_text(
                    chunk_text,
                    cell_type
                )

                chunk_info = {
                    "chunk_id": global_chunk_id,

                    "source": source,
                    "source_type": source_type,

                    "cell_number": cell_number,
                    "cell_type": cell_type,

                    "chunk_text": chunk_text,

                    "word_count": len(
                        chunk_text.split()
                    ),

                    "character_count": len(
                        chunk_text
                    ),

                    "preview": chunk_text[:300],

                    "num_units": len(
                        current_units
                    ),

                    "chunking_strategy":
                        "cell_aware_sentence_chunking",

                    "max_words": max_words,
                    "sentence_overlap": sentence_overlap
                }

                chunks_data.append(chunk_info)

                global_chunk_id += 1

                # ------------------------------------------------
                # Prepare overlap
                # ------------------------------------------------

                if sentence_overlap > 0:

                    overlap_units = (
                        current_units[-sentence_overlap:]
                    )

                else:

                    overlap_units = []

                current_units = overlap_units.copy()

                current_word_count = sum(
                    len(unit.split())
                    for unit in current_units
                )

                # ------------------------------------------------
                # If overlap itself prevents the new unit from
                # fitting, remove older overlap units.
                # ------------------------------------------------

                while (
                    current_units
                    and
                    current_word_count + unit_word_count
                    > max_words
                ):

                    removed_unit = current_units.pop(0)

                    current_word_count -= len(
                        removed_unit.split()
                    )

            # ----------------------------------------------------
            # Add current unit
            # ----------------------------------------------------

            current_units.append(unit)

            current_word_count += unit_word_count

        # --------------------------------------------------------
        # Add final chunk from the cell
        # --------------------------------------------------------

        if current_units:

            chunk_text = "\n".join(
                current_units
                if cell_type == "code"
                else current_units
            )

            chunk_text = clean_text(
                chunk_text,
                cell_type
            )

            chunk_info = {
                "chunk_id": global_chunk_id,

                "source": source,
                "source_type": source_type,

                "cell_number": cell_number,
                "cell_type": cell_type,

                "chunk_text": chunk_text,

                "word_count": len(
                    chunk_text.split()
                ),

                "character_count": len(
                    chunk_text
                ),

                "preview": chunk_text[:300],

                "num_units": len(
                    current_units
                ),

                "chunking_strategy":
                    "cell_aware_sentence_chunking",

                "max_words": max_words,
                "sentence_overlap": sentence_overlap
            }

            chunks_data.append(chunk_info)

            global_chunk_id += 1

    return chunks_data

In [49]:
chunks_data = chunk_notebook_with_metadata(
    cleaned_cells_data,
    max_words=250,
    sentence_overlap=2
)

print("Chunking completed.")
print("Total chunks created:", len(chunks_data))

Chunking completed.
Total chunks created: 230


In [50]:
first_chunk = chunks_data[0]

print("Chunk ID:", first_chunk["chunk_id"])
print("Source:", first_chunk["source"])
print("Source type:", first_chunk["source_type"])
print("Cell number:", first_chunk["cell_number"])
print("Cell type:", first_chunk["cell_type"])
print("Word count:", first_chunk["word_count"])
print("Character count:", first_chunk["character_count"])

print("\nChunk text:")
print(first_chunk["chunk_text"])

Chunk ID: 0
Source: Retrival_Ext_QnA_Complete.ipynb
Source type: ipynb
Cell number: 1
Cell type: markdown
Word count: 146
Character count: 981

Chunk text:
# From Extractive PDF QnA to Retrieval-Based Document QnA using Embeddings, FAISS, and BERT Reader

In the previous notebook, we already built a complete **Extractive PDF QnA system** using a BERT-style reader model.
That system covered:

- PDF reading
- Page-wise text extraction
- Text chunking with overlap
- Manual extractive QnA using `start_logits` and `end_logits`
- Answer span extraction
- Chunk-wise QnA
- Answer ranking
- Source tracking
- Failure case analysis

In this new notebook, we will not spend too much time reteaching those parts.
We will quickly rebuild the required pipeline because this is a fresh Colab notebook.
The main focus of this notebook is to move from:

$$
\text{Brute-force Extractive PDF QnA}
$$

to

$$
\text{Retrieval-Based Document QnA}
$$

The final goal is to build this architecture:

$$
\text{Question

In [51]:
# Calculate chunk statistics

chunk_word_counts = [
    chunk["word_count"]
    for chunk in chunks_data
]

chunk_character_counts = [
    chunk["character_count"]
    for chunk in chunks_data
]

chunk_unit_counts = [
    chunk["num_units"]
    for chunk in chunks_data
]


print("Total chunks:", len(chunks_data))

print("\nWord Count Statistics")
print("Minimum words in a chunk:", min(chunk_word_counts))
print("Maximum words in a chunk:", max(chunk_word_counts))
print(
    "Average words per chunk:",
    round(np.mean(chunk_word_counts), 2)
)
print(
    "Median words per chunk:",
    round(np.median(chunk_word_counts), 2)
)


print("\nUnit Count Statistics")
print("Minimum units in a chunk:", min(chunk_unit_counts))
print("Maximum units in a chunk:", max(chunk_unit_counts))
print(
    "Average units per chunk:",
    round(np.mean(chunk_unit_counts), 2)
)


print("\nCharacter Count Statistics")
print(
    "Minimum characters in a chunk:",
    min(chunk_character_counts)
)
print(
    "Maximum characters in a chunk:",
    max(chunk_character_counts)
)
print(
    "Average characters per chunk:",
    round(np.mean(chunk_character_counts), 2)
)

Total chunks: 230

Word Count Statistics
Minimum words in a chunk: 6
Maximum words in a chunk: 250
Average words per chunk: 73.64
Median words per chunk: 65.5

Unit Count Statistics
Minimum units in a chunk: 1
Maximum units in a chunk: 24
Average units per chunk: 3.2

Character Count Statistics
Minimum characters in a chunk: 83
Maximum characters in a chunk: 3627
Average characters per chunk: 651.63


In [52]:
# Load BGE-M3 from local Hugging Face cache

embedding_model_name = "BAAI/bge-m3"

start_time = time.time()

embedding_model = SentenceTransformer(
    embedding_model_name,
    device=device,
    local_files_only=True
)

end_time = time.time()

print("Local BGE-M3 loaded successfully.")
print("Model name:", embedding_model_name)
print("Device:", device)
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)
print(
    "Loading time:",
    round(end_time - start_time, 2),
    "seconds"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Local BGE-M3 loaded successfully.
Model name: BAAI/bge-m3
Device: cpu
Embedding dimension: 1024
Loading time: 1.39 seconds


/var/folders/k8/ncpzqds97lb86kwyp4d1byq00000gn/T/ipykernel_49654/1113361267.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [53]:
# Test embedding model on a small example

sample_sentences = [
    "Transformers use self-attention mechanisms.",
    "Self-attention helps models capture relationships between tokens.",
    "The weather is sunny today."
]

sample_embeddings = embedding_model.encode(
    sample_sentences,
    convert_to_numpy=True
)

print("Number of sample sentences:", len(sample_sentences))
print("Embedding matrix shape:", sample_embeddings.shape)
print("Embedding dimension:", sample_embeddings.shape[1])

Number of sample sentences: 3
Embedding matrix shape: (3, 1024)
Embedding dimension: 1024


In [54]:
def create_chunk_embeddings(
    chunks_data: List[Dict[str, Any]],
    embedding_model: SentenceTransformer,
    batch_size: int = 32,
    normalize_embeddings: bool = True
) -> np.ndarray:
    """
    Create embeddings for all Daedalus notebook chunks.

    Parameters
    ----------
    chunks_data : List[Dict[str, Any]]
        List of chunk dictionaries. Each dictionary must contain
        'chunk_text' and 'cell_type'.

    embedding_model : SentenceTransformer
        Loaded sentence-transformers embedding model.

    batch_size : int
        Number of chunks encoded at once.

    normalize_embeddings : bool
        Whether to normalize embeddings to unit length.
        Useful when using cosine similarity.

    Returns
    -------
    chunk_embeddings : np.ndarray
        Embedding matrix of shape
        (number_of_chunks, embedding_dimension).
    """

    if len(chunks_data) == 0:
        raise ValueError(
            "chunks_data is empty. "
            "Create chunks before generating embeddings."
        )

    chunk_texts = []

    for chunk in chunks_data:

        text = chunk.get("chunk_text", "")
        cell_type = chunk.get("cell_type", "markdown")

        # Clean using the appropriate cell-type rules
        text = clean_text(
            text,
            cell_type
        )

        if len(text.strip()) == 0:
            text = "empty chunk"

        chunk_texts.append(text)

    start_time = time.time()

    chunk_embeddings = embedding_model.encode(
        chunk_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
        show_progress_bar=True
    )

    end_time = time.time()

    # Use float32 for efficient vector storage
    chunk_embeddings = chunk_embeddings.astype("float32")

    print("Chunk embeddings created successfully.")
    print("Number of chunks:", len(chunks_data))
    print("Embedding matrix shape:", chunk_embeddings.shape)
    print("Embedding dimension:", chunk_embeddings.shape[1])
    print("Embedding dtype:", chunk_embeddings.dtype)
    print(
        "Embedding normalized:",
        normalize_embeddings
    )
    print(
        "Embedding time:",
        round(end_time - start_time, 2),
        "seconds"
    )

    return chunk_embeddings

In [55]:
chunk_embeddings = create_chunk_embeddings(
    chunks_data,
    embedding_model,
    batch_size=32,
    normalize_embeddings=True
)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Chunk embeddings created successfully.
Number of chunks: 230
Embedding matrix shape: (230, 1024)
Embedding dimension: 1024
Embedding dtype: float32
Embedding normalized: True
Embedding time: 50.17 seconds


In [56]:
# Check whether embeddings are normalized

embedding_norms = np.linalg.norm(chunk_embeddings, axis=1)

print("Minimum norm:", round(float(np.min(embedding_norms)), 4))
print("Maximum norm:", round(float(np.max(embedding_norms)), 4))
print("Average norm:", round(float(np.mean(embedding_norms)), 4))

Minimum norm: 1.0
Maximum norm: 1.0
Average norm: 1.0


In [57]:
# Store the embedding index inside each chunk's metadata.
# We do not store the full embedding vectors inside chunks_data
# because that would make the metadata unnecessarily large.

for idx, chunk in enumerate(chunks_data):
    chunk["embedding_index"] = idx

print("Embedding indices added to chunk metadata.")

print("\nExample chunk metadata keys:")
print(chunks_data[0].keys())

Embedding indices added to chunk metadata.

Example chunk metadata keys:
dict_keys(['chunk_id', 'source', 'source_type', 'cell_number', 'cell_type', 'chunk_text', 'word_count', 'character_count', 'preview', 'num_units', 'chunking_strategy', 'max_words', 'sentence_overlap', 'embedding_index'])


In [58]:
# Mini demo: compare similarity between a query and first few
# notebook chunks using direct NumPy similarity

demo_question = (
    "Why is brute-force question answering not scalable "
    "for large documents?"
)

demo_question_embedding = embedding_model.encode(
    [demo_question],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")


# Compare with the first 5 chunks using dot product
num_demo_chunks = min(5, len(chunks_data))

similarities = np.dot(
    chunk_embeddings[:num_demo_chunks],
    demo_question_embedding[0]
)


print("Demo question:")
print(demo_question)

print("\nSimilarity with first few chunks:")

for idx, score in enumerate(similarities):

    print(
        f"Chunk ID: {chunks_data[idx]['chunk_id']} | "
        f"Cell: {chunks_data[idx]['cell_number']} | "
        f"Cell Type: {chunks_data[idx]['cell_type']} | "
        f"Score: {score:.4f}"
    )

    print_wrapped(
        chunks_data[idx]["preview"],
        width=100
    )

    print("-" * 100)

Demo question:
Why is brute-force question answering not scalable for large documents?

Similarity with first few chunks:
Chunk ID: 0 | Cell: 1 | Cell Type: markdown | Score: 0.4390
# From Extractive PDF QnA to Retrieval-Based Document QnA using Embeddings, FAISS, and BERT Reader
In the previous notebook, we already built a complete **Extractive PDF QnA system** using a BERT-
style reader model. That system covered:  - PDF reading - Page-wise text extraction - Text chunking
wit
----------------------------------------------------------------------------------------------------
Chunk ID: 1 | Cell: 2 | Cell Type: markdown | Score: 0.5124
## How This Notebook Will Be Taught  This notebook has two different pacing styles.  ### Part A:
Fast Rebuild Mode  For the parts that students already know from the previous notebook, we will move
quickly. These include:  - Library installation - Imports - PDF upload - PDF reading - Page-wise
text
--------------------------------------------------------

In [59]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

In [60]:
def create_qdrant_collection(
    client: QdrantClient,
    collection_name: str,
    chunk_embeddings: np.ndarray
) -> None:
    """
    Create a Qdrant collection for Daedalus chunk embeddings.

    Parameters
    ----------
    client : QdrantClient
        Initialized Qdrant client.

    collection_name : str
        Name of the Qdrant collection.

    chunk_embeddings : np.ndarray
        Embedding matrix of shape
        (number_of_chunks, embedding_dimension).

    Returns
    -------
    None
        Creates the Qdrant collection.
    """

    if chunk_embeddings is None:
        raise ValueError(
            "chunk_embeddings cannot be None."
        )

    if len(chunk_embeddings.shape) != 2:
        raise ValueError(
            "chunk_embeddings must be a 2D matrix."
        )

    if chunk_embeddings.dtype != np.float32:
        chunk_embeddings = chunk_embeddings.astype("float32")

    num_chunks, embedding_dim = chunk_embeddings.shape

    if num_chunks == 0:
        raise ValueError(
            "No embeddings found. "
            "Create chunk embeddings before creating the Qdrant collection."
        )

    # ------------------------------------------------------------
    # Create Qdrant collection
    # ------------------------------------------------------------

    # FIX: create_collection() raises
    #   ValueError: Collection daedalus_chunks already exists
    # on every re-run, because ./daedalus_qdrant persists on disk.
    # Drop the stale collection first so the notebook is re-runnable.

    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)
        print("Dropped existing collection:", collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=embedding_dim,
            distance=Distance.COSINE
        )
    )

    print("Qdrant collection created successfully.")
    print("Collection name:", collection_name)
    print("Number of vectors:", num_chunks)
    print("Embedding dimension:", embedding_dim)
    print("Distance metric: COSINE")

In [61]:
QDRANT_PATH = "./daedalus_qdrant"

# FIX: a path-based (local) Qdrant store is exclusively locked by its
# client. Re-running this cell without closing the previous client
# raises:
#   "Storage folder ./daedalus_qdrant is already accessed by
#    another instance of Qdrant client."
# Close any client left over from an earlier run in this kernel first.

if "qdrant_client" in globals():
    try:
        qdrant_client.close()
        print("Closed previous Qdrant client.")
    except Exception as exc:
        print("Could not close previous client:", exc)

qdrant_client = QdrantClient(path=QDRANT_PATH)

print("Qdrant client connected at:", QDRANT_PATH)

Qdrant client connected at: ./daedalus_qdrant


In [62]:
collection_name = "daedalus_chunks"

create_qdrant_collection(
    qdrant_client,
    collection_name,
    chunk_embeddings
)

Dropped existing collection: daedalus_chunks
Qdrant collection created successfully.
Collection name: daedalus_chunks
Number of vectors: 230
Embedding dimension: 1024
Distance metric: COSINE


In [63]:
def upload_chunks_to_qdrant(
    qdrant_client: QdrantClient,
    collection_name: str,
    chunks_data: List[Dict[str, Any]],
    chunk_embeddings: np.ndarray,
    batch_size: int = 64
) -> None:
    """
    Upload Daedalus chunk embeddings and metadata to Qdrant.

    Each chunk embedding is stored together with its metadata
    as the Qdrant payload.
    """

    if len(chunks_data) == 0:
        raise ValueError("chunks_data is empty.")

    if len(chunk_embeddings) == 0:
        raise ValueError("chunk_embeddings is empty.")

    if len(chunks_data) != len(chunk_embeddings):
        raise ValueError(
            "Number of chunks and embeddings must be the same."
        )

    points = []

    for idx, (chunk, embedding) in enumerate(
        zip(chunks_data, chunk_embeddings)
    ):

        payload = {
            "chunk_id": chunk["chunk_id"],
            "source": chunk["source"],
            "source_type": chunk["source_type"],
            "cell_number": chunk["cell_number"],
            "cell_type": chunk["cell_type"],
            "chunk_text": chunk["chunk_text"],
            "word_count": chunk["word_count"],
            "character_count": chunk["character_count"],
            "preview": chunk["preview"],
            "num_units": chunk["num_units"],
            "chunking_strategy": chunk["chunking_strategy"],
            "max_words": chunk["max_words"],
            "sentence_overlap": chunk["sentence_overlap"],
            "embedding_index": idx
        }

        point = PointStruct(
            id=idx,
            vector=embedding.tolist(),
            payload=payload
        )

        points.append(point)

        # Upload in batches
        if len(points) >= batch_size:

            qdrant_client.upsert(
                collection_name=collection_name,
                points=points
            )

            points = []

    # Upload remaining points
    if points:

        qdrant_client.upsert(
            collection_name=collection_name,
            points=points
        )

    print("Chunks uploaded to Qdrant successfully.")
    print("Total vectors uploaded:", len(chunk_embeddings))

In [64]:
upload_chunks_to_qdrant(
    qdrant_client=qdrant_client,
    collection_name=collection_name,
    chunks_data=chunks_data,
    chunk_embeddings=chunk_embeddings
)

Chunks uploaded to Qdrant successfully.
Total vectors uploaded: 230


In [65]:
collection_info = qdrant_client.get_collection(
    collection_name
)

print("Collection:", collection_name)
print(
    "Vectors in collection:",
    collection_info.points_count
)

Collection: daedalus_chunks
Vectors in collection: 230


In [66]:
def retrieve_top_k_chunks(
    question: str,
    embedding_model: SentenceTransformer,
    qdrant_client: QdrantClient,
    collection_name: str,
    chunks_data: List[Dict[str, Any]],
    top_k: int = 5
) -> List[Dict[str, Any]]:
    """
    Retrieve the top-k most relevant chunks for a user question
    using BGE-M3 embeddings and Qdrant.

    Parameters
    ----------
    question : str
        User question.

    embedding_model : SentenceTransformer
        The same embedding model used to create chunk embeddings.

    qdrant_client : QdrantClient
        Initialized Qdrant client containing the chunk vectors.

    collection_name : str
        Name of the Qdrant collection.

    chunks_data : List[Dict[str, Any]]
        Chunk metadata list.

    top_k : int
        Number of chunks to retrieve.

    Returns
    -------
    retrieved_chunks : List[Dict[str, Any]]
        Retrieved chunks with similarity scores, ranks,
        and source metadata.
    """

    # ------------------------------------------------------------
    # Validate inputs
    # ------------------------------------------------------------

    if question is None or len(question.strip()) == 0:
        raise ValueError("Question cannot be empty.")

    if len(chunks_data) == 0:
        raise ValueError("chunks_data is empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # FIX: top_k used to be capped against len(chunks_data), the
    # in-memory list, which says nothing about what is actually in
    # Qdrant. Cap it against the collection, and refuse to run against
    # an empty index.
    #
    # This is the bug that made retrieval look broken: the first
    # retrieval call ran BEFORE the upload cell, queried an empty
    # collection, returned [] with no error, and printed nothing.

    collection_info = qdrant_client.get_collection(collection_name)
    points_count = collection_info.points_count

    if points_count == 0:
        raise RuntimeError(
            f"Qdrant collection '{collection_name}' contains 0 vectors. "
            "Upload the chunk embeddings before retrieving."
        )

    top_k = min(top_k, points_count)

    # ------------------------------------------------------------
    # Clean the question
    # ------------------------------------------------------------

    cleaned_question = clean_text(
        question,
        "markdown"
    )

    # ------------------------------------------------------------
    # Create question embedding
    # ------------------------------------------------------------

    question_embedding = embedding_model.encode(
        [cleaned_question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Convert NumPy array to a normal Python list
    query_vector = question_embedding[0].tolist()

    # ------------------------------------------------------------
    # Search Qdrant
    # ------------------------------------------------------------

    search_results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True
    ).points

    # ------------------------------------------------------------
    # Build retrieved chunk results
    # ------------------------------------------------------------

    retrieved_chunks = []

    for rank, result in enumerate(
        search_results,
        start=1
    ):

        payload = result.payload

        retrieved_chunk = {
            "rank": rank,
            "retrieval_score": float(result.score),

            "chunk_id": payload.get(
                "chunk_id",
                result.id
            ),

            "source": payload.get(
                "source",
                ""
            ),

            "source_type": payload.get(
                "source_type",
                ""
            ),

            "cell_number": payload.get(
                "cell_number",
                None
            ),

            "cell_type": payload.get(
                "cell_type",
                ""
            ),

            "chunk_text": payload.get(
                "chunk_text",
                ""
            ),

            "word_count": payload.get(
                "word_count",
                0
            ),

            "character_count": payload.get(
                "character_count",
                0
            ),

            "preview": payload.get(
                "preview",
                ""
            )
        }

        # Preserve optional metadata
        if "num_units" in payload:
            retrieved_chunk["num_units"] = payload["num_units"]

        if "chunking_strategy" in payload:
            retrieved_chunk["chunking_strategy"] = (
                payload["chunking_strategy"]
            )

        retrieved_chunks.append(
            retrieved_chunk
        )

    return retrieved_chunks

In [67]:
question = (
    "Why is brute-force question answering "
    "not scalable for large documents?"
)

retrieved_chunks = retrieve_top_k_chunks(
    question=question,
    embedding_model=embedding_model,
    qdrant_client=qdrant_client,
    collection_name=collection_name,
    chunks_data=chunks_data,
    top_k=5
)

In [68]:
for chunk in retrieved_chunks:

    print(
        f"Rank: {chunk['rank']} | "
        f"Score: {chunk['retrieval_score']:.4f} | "
        f"Cell: {chunk['cell_number']} | "
        f"Type: {chunk['cell_type']}"
    )

    print_wrapped(
        chunk["preview"],
        width=100
    )

    print("-" * 100)

Rank: 1 | Score: 0.6918 | Cell: 54 | Type: markdown
# Section 6: Why Brute-force QnA Is Not Scalable  Until now, our system follows this flow:  ```text
Question → Every Chunk → BERT Reader → Ranked Answers ``` This is called a brute-force QnA approach.
It works because the reader checks every possible chunk. But this approach has a serious problem: `
----------------------------------------------------------------------------------------------------
Rank: 2 | Score: 0.6516 | Cell: 64 | Type: markdown
## Observation / Interpretation  From the scaling table, we can see the main issue. In brute-force
QnA:  ```text Number of reader calls = Number of chunks ``` So if we have: ``` 10,000 chunks ```
then the BERT-style reader runs: ``` 10,000 times for one question ``` But in retrieval-based QnA,
if we
----------------------------------------------------------------------------------------------------
Rank: 3 | Score: 0.6426 | Cell: 67 | Type: markdown
## Concept Check  1. Why does brute-forc

In [69]:
question = (
    "What is the role of embeddings in the retrieval process?"
)

retrieved_chunks = retrieve_top_k_chunks(
    question=question,
    embedding_model=embedding_model,
    qdrant_client=qdrant_client,
    collection_name=collection_name,
    chunks_data=chunks_data,
    top_k=5
)

In [70]:
for chunk in retrieved_chunks:

    print(
        f"Rank: {chunk['rank']} | "
        f"Score: {chunk['retrieval_score']:.4f} | "
        f"Cell: {chunk['cell_number']} | "
        f"Type: {chunk['cell_type']}"
    )

    print_wrapped(
        chunk["preview"],
        width=100
    )

    print("-" * 100)

Rank: 1 | Score: 0.6471 | Cell: 146 | Type: markdown
## Bridge to Next Section  We now have a complete retrieval-based QnA system. But there is still one
limitation. Our manual retriever compares the question embedding with every chunk embedding one by
one. The retrieval step currently does this:  ```text Question Embedding → Compare with all chunk em
----------------------------------------------------------------------------------------------------
Rank: 2 | Score: 0.6449 | Cell: 79 | Type: markdown
## Why Embeddings Help Retrieval  Suppose we ask:  ```text "What are the disadvantages of the
method?" ``` A keyword-based system may search for the exact word: ``` disadvantages ``` But if the
document uses: ``` limitations ``` then keyword search may miss the relevant chunk. Embeddings help
becaus
----------------------------------------------------------------------------------------------------
Rank: 3 | Score: 0.6319 | Cell: 4 | Type: markdown
## Important Distinctions  Throughout th

In [71]:
from sentence_transformers import SentenceTransformer, CrossEncoder

In [72]:
# Load BGE reranker model

reranker_model_name = "BAAI/bge-reranker-v2-m3"

start_time = time.time()

reranker = CrossEncoder(
    reranker_model_name,
    max_length=512,
    device=device
)

end_time = time.time()

print("Reranker loaded successfully.")
print("Model name:", reranker_model_name)
print("Device:", device)
print(
    "Loading time:",
    round(end_time - start_time, 2),
    "seconds"
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker loaded successfully.
Model name: BAAI/bge-reranker-v2-m3
Device: cpu
Loading time: 12.11 seconds


In [73]:
def rerank_chunks(
    question: str,
    retrieved_chunks: List[Dict[str, Any]],
    reranker: CrossEncoder,
    final_top_k: int = 5
) -> List[Dict[str, Any]]:
    """
    Rerank retrieved notebook chunks using a cross-encoder reranker.

    Parameters
    ----------
    question : str
        User question.

    retrieved_chunks : List[Dict[str, Any]]
        Chunks retrieved by the vector search stage.

    reranker : CrossEncoder
        Loaded cross-encoder reranking model.

    final_top_k : int
        Number of chunks to return after reranking.

    Returns
    -------
    reranked_chunks : List[Dict[str, Any]]
        Reranked chunks with reranker scores and final ranks.
    """

    if question is None or len(question.strip()) == 0:
        raise ValueError("Question cannot be empty.")

    if len(retrieved_chunks) == 0:
        return []

    if final_top_k <= 0:
        raise ValueError(
            "final_top_k must be greater than 0."
        )

    final_top_k = min(
        final_top_k,
        len(retrieved_chunks)
    )

    # ------------------------------------------------------------
    # Clean question
    # ------------------------------------------------------------

    cleaned_question = clean_text(
        question,
        "markdown"
    )

    # ------------------------------------------------------------
    # Create question-chunk pairs
    # ------------------------------------------------------------

    pairs = []

    for chunk in retrieved_chunks:

        chunk_text = chunk.get(
            "chunk_text",
            ""
        )

        pairs.append(
            [
                cleaned_question,
                chunk_text
            ]
        )

    # ------------------------------------------------------------
    # Calculate reranker scores
    # ------------------------------------------------------------

    start_time = time.time()

    reranker_scores = reranker.predict(
        pairs,
        show_progress_bar=True
    )

    end_time = time.time()

    # ------------------------------------------------------------
    # Attach reranker scores
    # ------------------------------------------------------------

    scored_chunks = []

    for chunk, score in zip(
        retrieved_chunks,
        reranker_scores
    ):

        reranked_chunk = chunk.copy()

        reranked_chunk["reranker_score"] = float(
            score
        )

        scored_chunks.append(
            reranked_chunk
        )

    # ------------------------------------------------------------
    # Sort by reranker score
    # ------------------------------------------------------------

    scored_chunks.sort(
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    # ------------------------------------------------------------
    # Assign final ranks
    # ------------------------------------------------------------

    reranked_chunks = []

    for rank, chunk in enumerate(
        scored_chunks[:final_top_k],
        start=1
    ):

        chunk["reranker_rank"] = rank

        reranked_chunks.append(
            chunk
        )

    print("Reranking completed.")
    print("Candidates reranked:", len(retrieved_chunks))
    print("Final chunks:", len(reranked_chunks))
    print(
        "Reranking time:",
        round(end_time - start_time, 2),
        "seconds"
    )

    return reranked_chunks

In [74]:
retrieved_chunks = retrieve_top_k_chunks(
    question=question,
    embedding_model=embedding_model,
    qdrant_client=qdrant_client,
    collection_name=collection_name,
    chunks_data=chunks_data,
    top_k=10
)

In [75]:
reranked_chunks = rerank_chunks(
    question=question,
    retrieved_chunks=retrieved_chunks,
    reranker=reranker,
    final_top_k=5
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Reranking completed.
Candidates reranked: 10
Final chunks: 5
Reranking time: 5.11 seconds


In [76]:
for chunk in reranked_chunks:

    print(
        f"Final Rank: {chunk['reranker_rank']} | "
        f"Reranker Score: {chunk['reranker_score']:.4f} | "
        f"Vector Rank: {chunk['rank']} | "
        f"Vector Score: {chunk['retrieval_score']:.4f} | "
        f"Cell: {chunk['cell_number']} | "
        f"Type: {chunk['cell_type']}"
    )

    print_wrapped(
        chunk["preview"],
        width=100
    )

    print("-" * 100)

Final Rank: 1 | Reranker Score: 0.9263 | Vector Rank: 2 | Vector Score: 0.6449 | Cell: 79 | Type: markdown
## Why Embeddings Help Retrieval  Suppose we ask:  ```text "What are the disadvantages of the
method?" ``` A keyword-based system may search for the exact word: ``` disadvantages ``` But if the
document uses: ``` limitations ``` then keyword search may miss the relevant chunk. Embeddings help
becaus
----------------------------------------------------------------------------------------------------
Final Rank: 2 | Reranker Score: 0.5307 | Vector Rank: 5 | Vector Score: 0.6128 | Cell: 122 | Type: markdown
## Instructor-Only Answers  1. The input is a question, document chunks, chunk embeddings, and an
embedding model. 2. The output is a ranked list of relevant chunks with cosine similarity scores and
metadata. 3. We create chunk embeddings first so that the document becomes searchable in vector
form.
----------------------------------------------------------------------------------

In [77]:
# Test Daedalus retrieval with a new question

test_question = "What is RAG?"

# ------------------------------------------------------------
# Stage 1: Retrieve candidate chunks from Qdrant
# ------------------------------------------------------------

retrieved_chunks = retrieve_top_k_chunks(
    question=test_question,
    embedding_model=embedding_model,
    qdrant_client=qdrant_client,
    collection_name=collection_name,
    chunks_data=chunks_data,
    top_k=10
)

# ------------------------------------------------------------
# Stage 2: Rerank retrieved chunks
# ------------------------------------------------------------

reranked_chunks = rerank_chunks(
    question=test_question,
    retrieved_chunks=retrieved_chunks,
    reranker=reranker,
    final_top_k=5
)

# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print("=" * 100)
print("QUESTION")
print("=" * 100)
print(test_question)

print("\n" + "=" * 100)
print("RERANKED RESULTS")
print("=" * 100)

for chunk in reranked_chunks:

    print(
        f"\nFinal Rank: {chunk['reranker_rank']} | "
        f"Reranker Score: {chunk['reranker_score']:.4f}"
    )

    print(
        f"Vector Rank: {chunk['rank']} | "
        f"Vector Score: {chunk['retrieval_score']:.4f}"
    )

    print(
        f"Source: {chunk['source']} | "
        f"Cell: {chunk['cell_number']} | "
        f"Type: {chunk['cell_type']}"
    )

    print("\nChunk:")
    print_wrapped(
        chunk["chunk_text"],
        width=100
    )

    print("-" * 100)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Reranking completed.
Candidates reranked: 10
Final chunks: 5
Reranking time: 3.46 seconds
QUESTION
What is RAG?

RERANKED RESULTS

Final Rank: 1 | Reranker Score: 0.5110
Vector Rank: 1 | Vector Score: 0.4845
Source: Retrival_Ext_QnA_Complete.ipynb | Cell: 141 | Type: markdown

Chunk:
## Modern AI Systems Connection  This Retriever + Reader idea is the foundation of many modern
document QnA and RAG systems. In modern RAG systems, the retriever often finds relevant chunks and
then a generator model writes the final response. A simplified RAG flow is:  ```text Question    ↓
Retriever    ↓ Relevant Chunks    ↓ LLM / Generator    ↓ Final Response ``` Our current system is
slightly different: ``` Question    ↓ Retriever    ↓ Relevant Chunks    ↓ BERT Reader    ↓ Extracted
Answer Span ``` So our current system is not generative RAG yet. It is a retrieval-based extractive
QnA system. But the core idea is the same: ``` Retrieve useful context before answering. ``` In more
advanced systems, retr

In [78]:
# Connect to locally running Qwen3-8B through Ollama

import requests
import time

ollama_url = "http://localhost:11434"
generation_model_name = "qwen3:8b"

start_time = time.time()

# Check Ollama connection
response = requests.get(
    f"{ollama_url}/api/version",
    timeout=10
)

response.raise_for_status()

end_time = time.time()

print("Ollama connected successfully.")
print("Model name:", generation_model_name)
print("Ollama URL:", ollama_url)
print("Ollama version:", response.json()["version"])
print("Connection time:", round(end_time - start_time, 2), "seconds")

Ollama connected successfully.
Model name: qwen3:8b
Ollama URL: http://localhost:11434
Ollama version: 0.32.3
Connection time: 0.04 seconds


In [79]:
# Check available Ollama models

response = requests.get(
    f"{ollama_url}/api/tags",
    timeout=10
)

response.raise_for_status()

models = response.json()["models"]

print("Available Ollama models:")

for model in models:
    print("-", model["name"])

Available Ollama models:
- qwen3:4b
- qwen3:8b
- llama3.2:latest


In [80]:
# Test Qwen3-8B generation

test_prompt = """
What is the difference between retrieval failure and generation failure?
"""

start_time = time.time()

response = requests.post(
    f"{ollama_url}/api/chat",
    json={
        "model": generation_model_name,
        "messages": [
            {
                "role": "user",
                "content": test_prompt
            }
        ],
        "stream": False
    },
    timeout=300
)

response.raise_for_status()

result = response.json()

end_time = time.time()

print("Qwen3-8B response:")
print(result["message"]["content"])

print(
    "\nGeneration time:",
    round(end_time - start_time, 2),
    "seconds"
)

Qwen3-8B response:
**Retrieval Failure** and **Generation Failure** are distinct concepts, often encountered in psychology, linguistics, and cognitive science. Here's a clear breakdown of their differences:

---

### **1. Retrieval Failure**
- **Definition**: The inability to access information that is **already stored** in memory.  
- **Key Mechanism**: This occurs during the **retrieval phase** of memory, where the brain fails to locate or retrieve stored information despite its presence.  
- **Examples**:  
  - Forgetting a name or fact you studied (e.g., "I know I learned this, but I can't recall it").  
  - Difficulty recalling a word during a conversation (e.g., "I know the word, but it's on the tip of my tongue").  
- **Causes**:  
  - Interference (e.g., similar memories conflicting).  
  - Lack of effective retrieval cues.  
  - Decay of memory traces over time.  
- **Context**: Common in memory studies, such as in the **levels of processing theory** or **encoding specificity*

In [81]:
def build_evidence_context(
    chunks: List[Dict[str, Any]]
) -> str:
    """
    Format reranked chunks into the evidence block given to the LLM.

    Parameters
    ----------
    chunks : List[Dict[str, Any]]
        Reranked chunks for ONE question.

    Returns
    -------
    str
        Evidence context string.
    """

    evidence_parts = []

    for chunk in chunks:

        evidence_parts.append(
            f"""
[Source: {chunk.get('source', 'Unknown')}]
[Cell: {chunk.get('cell_number', 'Unknown')}]
[Cell Type: {chunk.get('cell_type', 'Unknown')}]

{chunk['chunk_text']}
"""
        )

    return "\n".join(evidence_parts)

In [82]:
def build_generation_prompt(
    question: str,
    evidence_context: str
) -> str:
    """
    Build the grounded-answer prompt.

    The question and its evidence are passed together so they cannot
    drift apart.
    """

    return f"""

You are Daedalus, an AI/ML interview preparation assistant.

Your task is to answer the user's question STRICTLY from the
provided study material.

IMPORTANT RULES:

1. The study material is the ONLY source of truth.
2. Do NOT use your general knowledge to answer the question.
3. Do NOT reinterpret the question using another field or domain.
4. Preserve the meaning and terminology used in the study material.
5. If the study material defines a concept, use that definition.
6. Do NOT invent examples, mechanisms, definitions, or explanations
   that are not supported by the study material.
7. If the evidence is insufficient, explicitly say that the
   provided study material does not contain enough information.
8. Give a concise, interview-ready answer.

USER QUESTION:

{question}

PROVIDED STUDY MATERIAL:

{evidence_context}

Now answer the user's question using ONLY the provided study material.

"""

In [83]:
def answer_question(
    question: str,
    embedding_model: SentenceTransformer,
    qdrant_client: QdrantClient,
    collection_name: str,
    chunks_data: List[Dict[str, Any]],
    reranker: CrossEncoder,
    retrieve_top_k: int = 10,
    final_top_k: int = 5,
    temperature: float = 0.1
) -> Dict[str, Any]:
    """
    Run the full RAG pipeline for a single question.

    FIX: previously the question was set in one cell while the evidence
    context was built from `reranked_chunks` left over from an EARLIER
    question. The prompt asked question X and handed the model evidence
    for question Y, so the retrieval stage was effectively bypassed.

    Retrieval, reranking, evidence building and generation now all take
    the same `question` argument, so they cannot drift apart.

    Returns
    -------
    Dict[str, Any]
        question, retrieved_chunks, reranked_chunks, evidence_context,
        prompt, answer, generation_time.
    """

    if question is None or len(question.strip()) == 0:
        raise ValueError("Question cannot be empty.")

    # -------- Stage 1: vector retrieval --------
    retrieved_chunks = retrieve_top_k_chunks(
        question=question,
        embedding_model=embedding_model,
        qdrant_client=qdrant_client,
        collection_name=collection_name,
        chunks_data=chunks_data,
        top_k=retrieve_top_k
    )

    # -------- Stage 2: reranking --------
    reranked_chunks = rerank_chunks(
        question=question,
        retrieved_chunks=retrieved_chunks,
        reranker=reranker,
        final_top_k=final_top_k
    )

    if len(reranked_chunks) == 0:
        raise RuntimeError(
            "No chunks survived retrieval and reranking. "
            "Check that the Qdrant collection is populated."
        )

    # -------- Stage 3: evidence for THIS question --------
    evidence_context = build_evidence_context(reranked_chunks)

    prompt = build_generation_prompt(question, evidence_context)

    # -------- Stage 4: grounded generation --------
    start_time = time.time()

    response = requests.post(
        f"{ollama_url}/api/chat",
        json={
            "model": generation_model_name,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "think": False,
            "stream": False,
            "options": {
                "temperature": temperature
            }
        },
        timeout=300
    )

    response.raise_for_status()

    generation_time = time.time() - start_time

    answer = response.json()["message"]["content"]

    return {
        "question": question,
        "retrieved_chunks": retrieved_chunks,
        "reranked_chunks": reranked_chunks,
        "evidence_context": evidence_context,
        "prompt": prompt,
        "answer": answer,
        "generation_time": generation_time
    }

In [97]:
test_question = (
    "How does create_overlapping_chunks_from_pages determine the starting position of the next chunk?"
)

print("Question:")
print(test_question)

result = answer_question(
    question=test_question,
    embedding_model=embedding_model,
    qdrant_client=qdrant_client,
    collection_name=collection_name,
    chunks_data=chunks_data,
    reranker=reranker,
    retrieve_top_k=10,
    final_top_k=5
)

Question:
How does create_overlapping_chunks_from_pages determine the starting position of the next chunk?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Reranking completed.
Candidates reranked: 10
Final chunks: 5
Reranking time: 5.26 seconds


In [98]:
print("=" * 100)
print("QWEN3-8B RESPONSE")
print("=" * 100)
print(result["answer"])

print(
    "\nGeneration time:",
    round(result["generation_time"], 2),
    "seconds"
)

QWEN3-8B RESPONSE
The `create_overlapping_chunks_from_pages` function determines the starting position of the next chunk by incrementing the `start` variable by `chunk_size - overlap` after each chunk is created. This ensures that each subsequent chunk overlaps with the previous one by the specified number of words (`overlap`).

Generation time: 8.17 seconds


In [99]:
print("=" * 100)
print("EVIDENCE ACTUALLY USED FOR THIS QUESTION")
print("=" * 100)

for i, chunk in enumerate(result["reranked_chunks"], 1):
    print("\n" + "=" * 100)
    print(f"CHUNK {i}")
    print("=" * 100)
    print("SOURCE:", chunk.get("source", "Unknown"))
    print("CELL:", chunk.get("cell_number", "Unknown"))
    print("CELL TYPE:", chunk.get("cell_type", "Unknown"))
    print("RERANKER SCORE:", round(chunk["reranker_score"], 4))
    print("\nTEXT:")
    print(chunk["chunk_text"])

EVIDENCE ACTUALLY USED FOR THIS QUESTION

CHUNK 1
SOURCE: Retrival_Ext_QnA_Complete.ipynb
CELL: 27
CELL TYPE: code
RERANKER SCORE: 0.9549

TEXT:
def create_overlapping_chunks_from_pages(
    pages: List[Dict[str, Any]],
    chunk_size: int = 180,
    overlap: int = 40
) -> List[Dict[str, Any]]:
    """
    Creates overlapping word-based chunks from page-wise PDF text.

    Args:
        pages:
            List of dictionaries containing page_number and text.

        chunk_size:
            Number of words in each chunk.

        overlap:
            Number of words repeated between consecutive chunks.

    Returns:
        List of chunk dictionaries with metadata.
    """

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size.")

    chunks = []
    global_chunk_id = 0

    for page in pages:
        page_number = page["page_number"]
        text = page["text"]
        words = text.split()

        if len(words) == 0:
            continue

   

In [96]:
print("=" * 100)
print("RERANKED CHUNK SUMMARY")
print("=" * 100)

print("Question:", result["question"])
print("Number of chunks:", len(result["reranked_chunks"]))

for i, chunk in enumerate(result["reranked_chunks"], 1):
    print(f"\nCHUNK {i}")
    print("Source:", chunk.get("source", "Unknown"))
    print("Cell:", chunk.get("cell_number", "Unknown"))
    print("Cell Type:", chunk.get("cell_type", "Unknown"))
    print("Reranker score:", round(chunk["reranker_score"], 4))
    print("Text length:", len(chunk.get("chunk_text", "")))

RERANKED CHUNK SUMMARY
Question: Why does the chunking function use an overlap of 40 words between consecutive chunks?
Number of chunks: 5

CHUNK 1
Source: Retrival_Ext_QnA_Complete.ipynb
Cell: 27
Cell Type: code
Reranker score: 0.7451
Text length: 1559

CHUNK 2
Source: Retrival_Ext_QnA_Complete.ipynb
Cell: 28
Cell Type: code
Reranker score: 0.2885
Text length: 185

CHUNK 3
Source: Retrival_Ext_QnA_Complete.ipynb
Cell: 44
Cell Type: code
Reranker score: 0.0514
Text length: 1570

CHUNK 4
Source: Retrival_Ext_QnA_Complete.ipynb
Cell: 59
Cell Type: code
Reranker score: 0.0097
Text length: 936

CHUNK 5
Source: Retrival_Ext_QnA_Complete.ipynb
Cell: 56
Cell Type: markdown
Reranker score: 0.0087
Text length: 645
